In [0]:
# PIPELINE OLIST - CAMADA GOLD
# Objetivo: Construir tabelas dimensionais, fatos e visões analíticas para
#           responder às perguntas estratégicas de Finanças, Logística e Comercial.

from pyspark.sql.functions import col, datediff, when, count, sum as _sum, avg, round, current_timestamp, lit

# 1. Criação do Database Gold
spark.sql("CREATE DATABASE IF NOT EXISTS gold_olist")
print("Database 'gold_olist' criado/verificado com sucesso.")

# ==============================================================================
# TABELA FATO ENRIQUECIDA (BASE PARA OS DATA MARTS)
# ==============================================================================
print("Gerando tabela fato enriquecida: gold_vendas_detalhadas...")

df_pedidos = spark.table("silver_olist.silver_orders")
df_itens = spark.table("silver_olist.silver_order_items")
df_clientes = spark.table("silver_olist.silver_customers")
df_vendedores = spark.table("silver_olist.silver_sellers")

# Join principal entre Pedidos, Itens, Clientes e Vendedores
df_fato_vendas = df_pedidos.join(df_itens, on="order_id", how="inner") \
    .join(df_clientes, on="customer_id", how="left") \
    .join(df_vendedores, on="seller_id", how="left") \
    .select(
        df_pedidos["order_id"],
        df_pedidos["customer_id"],
        df_itens["seller_id"],
        df_itens["product_id"],
        df_pedidos["order_status"],
        df_pedidos["order_purchase_timestamp"],
        df_pedidos["order_approved_at"],
        df_pedidos["order_delivered_customer_date"],
        df_pedidos["order_estimated_delivery_date"],
        
        # Métrica de cálculo logístico (dias para entrega e flag de atraso)
        datediff(df_pedidos["order_delivered_customer_date"], df_pedidos["order_purchase_timestamp"]).alias("dias_para_entrega"),
        when(df_pedidos["order_delivered_customer_date"] > df_pedidos["order_estimated_delivery_date"], 1).otherwise(0).alias("flg_atraso"),
        
        df_itens["price"],
        df_itens["freight_value"],
        (df_itens["price"] + df_itens["freight_value"]).alias("valor_total_item"),
        
        df_clientes["customer_state"],
        df_clientes["customer_city"],
        df_vendedores["seller_state"],
        df_vendedores["seller_city"],
        
        # Flag de venda local (cliente e vendedor no mesmo estado)
        when(df_clientes["customer_state"] == df_vendedores["seller_state"], 1).otherwise(0).alias("flg_venda_local")
    )

df_fato_vendas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_olist.gold_vendas_detalhadas")
print("Tabela fato 'gold_vendas_detalhadas' criada com sucesso.")

In [0]:
# ==============================================================================
# 1. DOMÍNIO FINANÇAS E MEIOS DE PAGAMENTO
# Responde Pergunta 1 e Pergunta 2 de Finanças
# ==============================================================================
print("Processando Data Mart: gold_metrics_financas...")

df_pagamentos = spark.table("silver_olist.silver_order_payments")

df_financas = df_pagamentos.groupBy("payment_type").agg(
    round(avg("payment_value"), 2).alias("ticket_medio_pagamento"),
    round(_sum("payment_value"), 2).alias("receita_total_pagamento"),
    count("order_id").alias("total_transacoes"),
    round(avg("payment_installments"), 2).alias("media_parcelas")
)

df_financas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_olist.gold_metrics_financas")


# ==============================================================================
# 2. DOMÍNIO LOGÍSTICA E EXPERIÊNCIA DO CLIENTE
# Responde Pergunta 1 e Pergunta 2 de Logística
# ==============================================================================
print("Processando Data Mart: gold_metrics_logistica...")

df_fato = spark.table("gold_olist.gold_vendas_detalhadas")

# Filtra apenas pedidos entregues para cálculo do SLA
df_logistica = df_fato.filter(col("order_status") == "delivered").groupBy("customer_state").agg(
    round(avg("dias_para_entrega"), 2).alias("tempo_medio_entrega_dias"),
    count("order_id").alias("total_pedidos_entregues"),
    _sum("flg_atraso").alias("total_pedidos_atrasados"),
    round((_sum("flg_atraso") / count("order_id")) * 100, 2).alias("pct_pedidos_atrasados")
)

df_logistica.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_olist.gold_metrics_logistica")


# ==============================================================================
# 3. DOMÍNIO DESEMPENHO COMERCIAL E COBERTURA DE VENDEDORES
# Responde Pergunta 1 e Pergunta 2 do Desempenho Comercial
# ==============================================================================
print("Processando Data Mart: gold_metrics_desempenho_comercial...")

df_comercial = df_fato.groupBy("customer_state").agg(
    count("order_id").alias("total_itens_comercializados"),
    round(_sum("valor_total_item"), 2).alias("receita_total_estado"),
    
    # Métrica de participação de vendedores locais no atendimento do estado
    _sum("flg_venda_local").alias("qtd_itens_vendedor_local"),
    round((_sum("flg_venda_local") / count("order_id")) * 100, 2).alias("pct_participacao_vendedor_local")
)

df_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_olist.gold_metrics_desempenho_comercial")

print("\n--- PROCESSAMENTO DA CAMADA GOLD FINALIZADO COM SUCESSO! ---")